<a href="https://colab.research.google.com/github/DarleneJD/ACOPF/blob/main/curvasvoltvartrifasico_tempo_loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**DATA: 16/03/2026**
**REDE: 13 BARRAS DE BAIXA TENSÃO TRIFÁSICA DESBALANCEADA**

**FUNÇÃO:** avaliar o sistema com controle Volt-Var por meio da curva dos inversores.

**CURVA:** curva geral IEEE 1547.

**OBJETIVO:** minimizar as perdas elétricas por meio da função objetivo.

### **MODELO TRIFÁSICO DESBALANCEADO**

Rede radial de baixa tensão com 5 barras (400 V / 250 kVA), conectada a uma rede de média tensão de 20 kV por meio de um transformador. O sistema é modelado como **trifásico desbalanceado por fase**, de modo que cada barra possui variáveis elétricas independentes para as fases (a), (b) e (c), permitindo representar explicitamente assimetrias de carga, geração fotovoltaica e tensão.

As barras 1, 2 e 4 possuem **carga e geração fotovoltaica trifásicas**, com a potência total dividida igualmente entre as três fases. As barras 3 e 5 possuem **carga e geração fotovoltaica monofásicas**, conectadas a uma fase específica, enquanto as demais fases dessas barras permanecem sem injeção de potência ativa e reativa associada a esses elementos.

O problema é formulado como um **OPF (Optimal Power Flow)** na forma de **MINLP**, implementado em **Pyomo** e resolvido com o **BONMIN**. A modelagem elétrica utiliza equações do tipo **DistFlow por fase**, com variáveis de fluxo ativo, fluxo reativo, corrente ao quadrado e magnitude de tensão ao quadrado para cada ramo e fase da rede. Nesta formulação, cada fase é representada separadamente, permitindo capturar o comportamento desbalanceado do sistema.

A **função objetivo** minimiza as **perdas totais ativas da rede**, calculadas pela soma das perdas em todos os ramos e em todas as fases.

O controle **Volt-Var**, conforme a **IEEE 1547-2018**, é implementado **por barra e por fase**, ou seja, cada inversor responde à tensão local da fase à qual está conectado. A curva Volt-Var é representada com variáveis binárias e restrições do tipo **Big-M**, considerando cinco regiões operativas delimitadas pelos pontos:

* ($V_{1}$ = 0.92) pu
* ($V_{2}$ = 0.98) pu
* ($V_{3}$ = 1.02) pu
* ($V_{4}$ = 1.08) pu

Com isso, cada unidade fotovoltaica pode injetar ou absorver potência reativa de acordo com a tensão local da fase, respeitando seus limites de capacidade reativa.

As bases adotadas no sistema são:

* **potência base trifásica:** ($S_{base}$ = 250 kVA)
* **potência base por fase:** ($S_{base,1\phi}$ = $\frac{250}{3}$ kVA)
* **tensão base na baixa tensão:** ($V_{base}$ = 400  V) fase-fase

Nesta versão, o modelo representa a rede como **trifásica desbalanceada por fase**, com controle Volt-Var individual por fase, tornando a formulação coerente com a presença simultânea de elementos trifásicos e monofásicos ao longo do alimentador.



# Bibliotecas




In [26]:
%%capture
import sys
import os

if 'google.colab' in sys.modules:
    !pip install idaes-pse --pre
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'


In [27]:



# import shutil
# import sys
# import os.path

# if not shutil.which("pyomo"):
#     !pip install -q pyomo
#     assert(shutil.which("pyomo"))

# if not (shutil.which("ipopt") or os.path.isfile("ipopt")):
#     if "google.colab" in sys.modules:
#         !wget -N -q "https://matematica.unipv.it/gualandi/solvers/ipopt-linux64.zip"
#         !unzip -o -q ipopt-linux64
#     else:
#         try:
#             !conda install -c conda-forge ipopt
#         except:
#             pass


In [28]:
!rm -rf ACOPF
!git clone https://github.com/DarleneJD/ACOPF.git

Cloning into 'ACOPF'...
remote: Enumerating objects: 223, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 223 (delta 39), reused 0 (delta 0), pack-reused 153 (from 1)
Receiving objects: 100% (223/223), 1.82 MiB | 14.58 MiB/s, done.
Resolving deltas: 100% (130/130), done.


In [29]:
import sys
import os
from re import I
import pyomo.environ as pyo
from pyomo.opt import SolverFactory
import pandas as pd
import math
import numpy as np
import matplotlib.pyplot as plt



# Dados de entrada

In [30]:

# ============================================================
# LEITURA
# ============================================================
base_path = "/content/ACOPF"
controle = f"{base_path}/volt-var.xlsx"

df_bus_bt = pd.read_excel(controle, sheet_name="bus_BT")
df_pv_bt = pd.read_excel(controle, sheet_name="bus_PV")
df_branches_bt = pd.read_excel(controle, sheet_name="branches_BT")


In [31]:

# padronização de colunas
df_bus_bt.columns = df_bus_bt.columns.str.strip()
df_pv_bt.columns = df_pv_bt.columns.str.strip()
df_branches_bt.columns = df_branches_bt.columns.str.strip()

# ============================================================
# CONSTANTES GERAIS
# ============================================================
PHASES = ["a", "b", "c"]
PHASE_MAP = {"1": "a", "2": "b", "3": "c"}

SLACK_BUS = "SOURCE"

FP_CARGA = 0.9
ANGULO_CARGA = np.arccos(FP_CARGA)

V_SLACK_PU = 1.02
vmin, vmax = 0.90, 1.20

FATOR_CARGA = 0.5
FATOR_PV = 1.0

# curva Volt-Var
V1, V2, V3, V4 = 0.92, 0.98, 1.02, 1.08
V1_2, V2_2, V3_2, V4_2 = V1**2, V2**2, V3**2, V4**2
M_bigm = 5.0

# base elétrica
PERCENT_R_TRAFO = 1.91
PERCENT_X_TRAFO = 3.51
S_base_3ph = 250.0
S_base_1ph = S_base_3ph / 3.0

# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================
def parse_conn(phases_value):
    """
    Retorna:
    - conn_type: 'three_phase' ou 'single_phase'
    - conn_phase: 'a','b','c' ou None
    """
    s = str(phases_value).strip()

    if s == "1,2,3":
        return "three_phase", None
    if s in PHASE_MAP:
        return "single_phase", PHASE_MAP[s]

    raise ValueError(f"Valor inválido em 'phases': {phases_value}")


def safe_float(x, default=0.0):
    if pd.isna(x):
        return default
    return float(x)


def dividir_carga_por_fase(total_kw, bus, phase):
    """
    Divide carga agregada da barra por fase.
    """
    bus_type = BUS_TYPE[bus]
    bus_phase = BUS_PHASE[bus]

    if bus_type == "three_phase":
        return total_kw / 3.0
    if bus_type == "single_phase":
        return total_kw if phase == bus_phase else 0.0

    raise ValueError(f"Tipo de barra inválido: {bus}")


def calc_limites_qpv_por_fase(bus, phase, p_pv_1f_kw):
    """
    Limites Q do inversor por fase, em kVAr.
    Usa os dados já carregados por barra-fase.
    """
    p_nom_1f = P_PV_NOMINAL.get((bus, phase), 0.0)
    q_nom_1f = Q_PV_RATED.get((bus, phase), 0.0)
    s_nom_1f = S_PV_NOMINAL.get((bus, phase), 0.0)

    if p_nom_1f <= 1e-12 or q_nom_1f <= 1e-12 or s_nom_1f <= 1e-12:
        return 0.0, 0.0

    p1 = 0.05 * p_nom_1f
    p2 = 0.20 * p_nom_1f

    if p_pv_1f_kw < p1:
        qmax = 0.0
    elif p_pv_1f_kw >= p2:
        qmax = q_nom_1f
    else:
        qmax = 2.2 * s_nom_1f * (p_pv_1f_kw / p_nom_1f)

    return -qmax, qmax

In [32]:
# Padronização das fases,
def parse_conn(phases_value):
    """
    Converte a coluna 'phases' para:
    - conn_type: 'three_phase' ou 'single_phase'
    - conn_phase: 'a','b','c' ou None
    """
    s = str(phases_value).strip()

    if s == "1,2,3":
        return "three_phase", None

    if s in PHASE_MAP:
        return "single_phase", PHASE_MAP[s]

    raise ValueError(f"Valor inválido em 'phases': {phases_value}")


def safe_float(x, default=0.0):
    if pd.isna(x):
        return default
    return float(x)

### BUS BT

In [33]:
# ============================================================
# BARRAS BT
# ============================================================
BUS_NAME_TO_N = {}
BUS_N_TO_NAME = {}

BUS_TYPE = {}
BUS_PHASE = {}
V_NOM_KV_FN = {}

P_CARGA_NOMINAL_BARRA = {}
Q_CARGA_NOMINAL_BARRA = {}

buses_bt = []

for _, row in df_bus_bt.iterrows():
    bus_n = int(row["N"])
    bus_name = str(row["name"]).strip()

    conn_type, conn_phase = parse_conn(row["phases"])

    BUS_NAME_TO_N[bus_name] = bus_n
    BUS_N_TO_NAME[bus_n] = bus_name

    BUS_TYPE[bus_name] = conn_type
    BUS_PHASE[bus_name] = conn_phase
    V_NOM_KV_FN[bus_name] = safe_float(row["v_nom_kv"])

    P_CARGA_NOMINAL_BARRA[bus_name] = safe_float(row["P_D"])
    Q_CARGA_NOMINAL_BARRA[bus_name] = safe_float(row["Q_D"])

    buses_bt.append(bus_name)

buses_bt = sorted(set(buses_bt))

### Planilha PV

In [34]:
# ============================================================
# PV POR BARRA-FASE
# ============================================================
df_pv_bt["Bus"] = df_pv_bt["Bus"].astype(str).str.strip()
df_pv_bt["phases"] = df_pv_bt["phases"].astype(str).str.strip()

PV_TYPE = {}
PV_PHASE = {}

P_PV_NOMINAL = {}
Q_PV_RATED = {}
S_PV_NOMINAL = {}

# inicializa todas as barras/fases com zero
for bus in buses_bt:
    PV_TYPE[bus] = None
    PV_PHASE[bus] = None
    for ph in PHASES:
        P_PV_NOMINAL[(bus, ph)] = 0.0
        Q_PV_RATED[(bus, ph)] = 0.0
        S_PV_NOMINAL[(bus, ph)] = 0.0

# sobrescreve onde houver PV
for _, row in df_pv_bt.iterrows():
    bus = row["Bus"]
    conn_type, conn_phase = parse_conn(row["phases"])

    PV_TYPE[bus] = conn_type
    PV_PHASE[bus] = conn_phase

    kva_total = safe_float(row["kva"])

    p_by_phase = {
        "a": safe_float(row["p_pv_1"]),
        "b": safe_float(row["p_pv_2"]),
        "c": safe_float(row["p_pv_3"]),
    }

    q_by_phase = {
        "a": safe_float(row["q_pv_1"]),
        "b": safe_float(row["q_pv_2"]),
        "c": safe_float(row["q_pv_3"]),
    }

    if conn_type == "three_phase":
        for ph in PHASES:
            P_PV_NOMINAL[(bus, ph)] = p_by_phase[ph]
            Q_PV_RATED[(bus, ph)] = q_by_phase[ph]
            S_PV_NOMINAL[(bus, ph)] = kva_total / 3.0
    else:
        for ph in PHASES:
            if ph == conn_phase:
                P_PV_NOMINAL[(bus, ph)] = p_by_phase[ph]
                Q_PV_RATED[(bus, ph)] = q_by_phase[ph]
                S_PV_NOMINAL[(bus, ph)] = kva_total

### Branches

In [35]:
# ============================================================
# RAMOS BT
# ============================================================
LINES_BT = []
BRANCH_TYPE = {}
BRANCH_PHASE = {}
R_LINE = {}
X_LINE = {}
IMAX_LINE = {}
LENGTH_M = {}

for _, row in df_branches_bt.iterrows():
    from_bus = str(row["l"]).strip()
    to_bus = str(row["k"]).strip()

    conn_type, conn_phase = parse_conn(row["phase"])

    LINES_BT.append((from_bus, to_bus))
    BRANCH_TYPE[(from_bus, to_bus)] = conn_type
    BRANCH_PHASE[(from_bus, to_bus)] = conn_phase

    R_LINE[(from_bus, to_bus)] = safe_float(row["R"])
    X_LINE[(from_bus, to_bus)] = safe_float(row["X"])
    IMAX_LINE[(from_bus, to_bus)] = safe_float(row["Imax"])
    LENGTH_M[(from_bus, to_bus)] = safe_float(row["Length_m"])

LINES_BT = list(dict.fromkeys(LINES_BT))

### Topologia

In [36]:
# ============================================================
# TOPOLOGIA GLOBAL
# ============================================================
from_set = {i for i, j in LINES_BT}
to_set = {j for i, j in LINES_BT}

ROOTS_BT = sorted(from_set - to_set)
print("Raízes BT:", ROOTS_BT)

TRAFO_CONN = [(SLACK_BUS, root) for root in ROOTS_BT]
ALL_CONN = TRAFO_CONN + LINES_BT

UPSTREAM = {}
DOWNSTREAM = {bus: [] for bus in buses_bt}
DOWNSTREAM[SLACK_BUS] = []

for i, j in LINES_BT:
    UPSTREAM[j] = (i, j)
    DOWNSTREAM[i].append((i, j))

for root in ROOTS_BT:
    UPSTREAM[root] = (SLACK_BUS, root)
    DOWNSTREAM[SLACK_BUS].append((SLACK_BUS, root))

Raízes BT: ['B010', 'B020', 'B030', 'B040', 'B050', 'B060', 'B070', 'B080', 'B090', 'B100', 'B110']


### Bases Elétricas

In [37]:
# ============================================================
# BASES ELÉTRICAS
# ============================================================
# assumindo v_nom_kv em fase-neutro
V_BT_LN = list(V_NOM_KV_FN.values())[0]
V_BT_LL = V_BT_LN * np.sqrt(3)

Z_base_Ohm = ((V_BT_LL ** 2) / S_base_3ph) * 1000

R_TRAFO_Ohm = (PERCENT_R_TRAFO / 100.0) * Z_base_Ohm
X_TRAFO_Ohm = (PERCENT_X_TRAFO / 100.0) * Z_base_Ohm

R_TRAFO_1ph_Ohm = R_TRAFO_Ohm / 3.0
X_TRAFO_1ph_Ohm = X_TRAFO_Ohm / 3.0

R_TRAFO_pu = R_TRAFO_1ph_Ohm / Z_base_Ohm
X_TRAFO_pu = X_TRAFO_1ph_Ohm / Z_base_Ohm

R_LINE_PU = {k: v / Z_base_Ohm for k, v in R_LINE.items()}
X_LINE_PU = {k: v / Z_base_Ohm for k, v in X_LINE.items()}

R_CONN_PU = dict(R_LINE_PU)
X_CONN_PU = dict(X_LINE_PU)

for root in ROOTS_BT:
    R_CONN_PU[(SLACK_BUS, root)] = R_TRAFO_pu
    X_CONN_PU[(SLACK_BUS, root)] = X_TRAFO_pu

#### Mascaras

In [38]:
# ============================================================
# MÁSCARAS DE FASES EXISTENTES
# ============================================================
def phase_exists_bus(bus, ph):
    if bus == SLACK_BUS:
        return True
    if BUS_TYPE[bus] == "three_phase":
        return True
    return BUS_PHASE[bus] == ph


def phase_exists_branch(i, j, ph):
    if (i, j) in TRAFO_CONN:
        return True
    if BRANCH_TYPE[(i, j)] == "three_phase":
        return True
    return BRANCH_PHASE[(i, j)] == ph

### Parametros barra/fase

In [39]:
# ============================================================
# PARÂMETROS POR BARRA-FASE
# ============================================================
P_carga_fase = {}
Q_carga_fase = {}
P_pv_fase = {}
Qmin_fase = {}
Qmax_fase = {}

for bus in buses_bt:
    p_carga_total = P_CARGA_NOMINAL_BARRA.get(bus, 0.0) * FATOR_CARGA
    q_carga_total = Q_CARGA_NOMINAL_BARRA.get(bus, 0.0) * FATOR_CARGA

    print(f"\nBarra {bus}")

    for ph in PHASES:
        # carga
        p_carga_1f = dividir_carga_por_fase(p_carga_total, bus, ph)
        q_carga_1f = dividir_carga_por_fase(q_carga_total, bus, ph)

        P_carga_fase[(bus, ph)] = p_carga_1f / S_base_1ph
        Q_carga_fase[(bus, ph)] = q_carga_1f / S_base_1ph

        # PV já vem por fase
        p_pv_1f = P_PV_NOMINAL.get((bus, ph), 0.0) * FATOR_PV
        P_pv_fase[(bus, ph)] = p_pv_1f / S_base_1ph

        qmin_1f, qmax_1f = calc_limites_qpv_por_fase(bus, ph, p_pv_1f)
        Qmin_fase[(bus, ph)] = qmin_1f / S_base_1ph
        Qmax_fase[(bus, ph)] = qmax_1f / S_base_1ph

        print(
            f"  fase {ph}: "
            f"Pcarga={p_carga_1f:8.3f} kW | "
            f"Qcarga={q_carga_1f:8.3f} kVAr | "
            f"Ppv={p_pv_1f:8.3f} kW | "
            f"Qpv±={qmax_1f:8.3f} kVAr"
        )


Barra B010
  fase a: Pcarga=   0.000 kW | Qcarga=   0.000 kVAr | Ppv=   0.000 kW | Qpv±=   0.000 kVAr
  fase b: Pcarga=   0.000 kW | Qcarga=   0.000 kVAr | Ppv=   0.000 kW | Qpv±=   0.000 kVAr
  fase c: Pcarga=   0.000 kW | Qcarga=   0.000 kVAr | Ppv=   0.000 kW | Qpv±=   0.000 kVAr

Barra B011
  fase a: Pcarga=  16.667 kW | Qcarga=   5.000 kVAr | Ppv=  25.261 kW | Qpv±=   0.000 kVAr
  fase b: Pcarga=  16.667 kW | Qcarga=   5.000 kVAr | Ppv=  25.261 kW | Qpv±=   0.000 kVAr
  fase c: Pcarga=  16.667 kW | Qcarga=   5.000 kVAr | Ppv=  25.261 kW | Qpv±=   0.000 kVAr

Barra B012
  fase a: Pcarga=  16.667 kW | Qcarga=   5.000 kVAr | Ppv=  25.261 kW | Qpv±=   0.000 kVAr
  fase b: Pcarga=  16.667 kW | Qcarga=   5.000 kVAr | Ppv=  25.261 kW | Qpv±=   0.000 kVAr
  fase c: Pcarga=  16.667 kW | Qcarga=   5.000 kVAr | Ppv=  25.261 kW | Qpv±=   0.000 kVAr

Barra B013
  fase a: Pcarga=  16.667 kW | Qcarga=   5.000 kVAr | Ppv=  25.261 kW | Qpv±=   0.000 kVAr
  fase b: Pcarga=  16.667 kW | Qcarga=   5

In [40]:
print("=" * 70)
print("SISTEMA TRIFÁSICO DESBALANCEADO POR FASE")
print("=" * 70)
print(f"V_BT_FF = {V_BT_LL*1000:.0f} V |")
print(f"S_base_3f = {S_base_3ph:.2f} kVA | S_base_1f = {S_base_1ph:.4f} kVA")
print(f"Z_base = {Z_base_Ohm:.4f} ohm")
print(f"R_TRAFO_pu = {R_TRAFO_pu:.6f}")
print(f"X_TRAFO_pu = {X_TRAFO_pu:.6f}")
print(f"R_LINE_PU = {R_LINE_PU}")
print(f"X_LINE_PU = {X_LINE_PU}")

SISTEMA TRIFÁSICO DESBALANCEADO POR FASE
V_BT_FF = 831 V |
S_base_3f = 250.00 kVA | S_base_1f = 83.3333 kVA
Z_base = 2.7648 ohm
R_TRAFO_pu = 0.006367
X_TRAFO_pu = 0.011700
R_LINE_PU = {('B010', 'B011'): np.float64(0.0006872106481481484), ('B011', 'B012'): np.float64(0.0006872106481481484), ('B012', 'B013'): np.float64(0.0006872106481481484), ('B013', 'B014'): np.float64(0.0006872106481481484), ('B020', 'B021'): np.float64(0.0006872106481481484), ('B021', 'B022'): np.float64(0.0006872106481481484), ('B022', 'B023'): np.float64(0.0006872106481481484), ('B023', 'B024'): np.float64(0.0006872106481481484), ('B030', 'B031'): np.float64(0.0006872106481481484), ('B031', 'B032'): np.float64(0.0006872106481481484), ('B032', 'B033'): np.float64(0.0006872106481481484), ('B033', 'B034'): np.float64(0.0006872106481481484), ('B040', 'B041'): np.float64(0.0006872106481481484), ('B041', 'B042'): np.float64(0.0006872106481481484), ('B042', 'B043'): np.float64(0.0006872106481481484), ('B043', 'B044'): np

# Modelo

In [41]:
# ============================================================
# CONFIGURAÇÃO DO HORIZONTE TEMPORAL
# ============================================================
HORAS = list(range(24))

HORA_SEM_PV = set(range(0, 7)) | set(range(19, 24))   # [0,7) U [19,24)
HORA_COM_PV = set(range(7, 19))                         # [7,18]

# Fração de Qmax permitida como variação máxima entre horas consecutivas.
# Exemplo: 0.30 → o inversor pode mudar no máximo 30% de Qmax por hora.
DELTA_Q_MAX_FRAC = 0.30

# Perfil de carga horário (fator multiplicativo sobre P_CARGA_NOMINAL_BARRA)
# Índice = hora do dia (0..23)
LOAD_PROFILE = [
    0.40, 0.35, 0.30, 0.28, 0.28, 0.32,
    0.45, 0.65, 0.75, 0.70, 0.65, 0.60,
    0.65, 0.60, 0.58, 0.60, 0.65, 0.80,
    1.00, 0.95, 0.90, 0.80, 0.65, 0.50,
]

# Perfil de geração PV horário (fator multiplicativo sobre P_PV_NOMINAL)
def pv_profile(h):
    if 7 <= h <= 18:
        return math.sin(math.pi * (h - 7) / 11)
    return 0.0

PV_PROFILE = [pv_profile(h) for h in HORAS]

# ============================================================
# MEMÓRIA INTER-HORA (estado Q_PV do instante anterior)
# Inicializado em zero: antes de t=0 não há geração nem injeção.
# ============================================================
Q_PV_anterior = {(bus, ph): 0.0 for bus in buses_bt for ph in PHASES}

# BP_anterior: breakpoints otimizados da hora anterior, indexados por (bus, ph)
# Inicializados nos valores IEEE 1547 — o solver parte desse ponto na hora 7h.
BP_anterior = {
    "V1": {},   # preenchido apos cada hora com PV resolvida
    "V2": {},
    "V3": {},
    "V4": {},
}

# Acumulador de resultados
resultados_lista = []

In [42]:
# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================
def pv_exists(bus, ph, tol=1e-9):
    return (
        abs(P_PV_NOMINAL.get((bus, ph), 0.0)) > tol
        or abs(Q_PV_RATED.get((bus, ph), 0.0)) > tol
        or abs(S_PV_NOMINAL.get((bus, ph), 0.0)) > tol
    )

def safe_val(x, default=np.nan):
    try:
        v = pyo.value(x)
        if v is None:
            return default
        return float(v)
    except Exception:
        return default

SOLVERS_NLP  = ['ipopt', 'couenne']
SOLVER_MINLP = 'bonmin'

def configurar_solver_nlp():
    for nome in SOLVERS_NLP:
        s = SolverFactory(nome)
        if s.available():
            s.options['max_iter'] = 10000
            s.options['tol']      = 1e-6
            return s, nome
    raise RuntimeError(f"Nenhum solver NLP disponível: {SOLVERS_NLP}")

def configurar_solver_minlp():
    s = SolverFactory(SOLVER_MINLP)
    if not s.available():
        raise RuntimeError(f"Solver MINLP '{SOLVER_MINLP}' não disponível.")
    s.options['max_iter']             = 10000
    s.options['tol']                  = 1e-4
    s.options['bonmin.bb_log_level']  = 0
    s.options['bonmin.nlp_log_level'] = 0
    return s, SOLVER_MINLP

In [43]:
# ============================================================
# LOOP TEMPORAL — um OPF por hora
# ============================================================
for t in HORAS:
    fator_carga_t = LOAD_PROFILE[t]
    fator_pv_t    = PV_PROFILE[t]

    print(f"\n{'='*70}")
    print(f"HORA t={t:02d}h  |  fator_carga={fator_carga_t:.2f}  |  fator_pv={fator_pv_t:.4f}")
    print(f"{'='*70}")

    # ----------------------------------------------------------
    # Recalcular parâmetros de carga e PV para esta hora
    # ----------------------------------------------------------
    P_carga_fase_t = {}
    Q_carga_fase_t = {}
    P_pv_fase_t    = {}
    Qmin_fase_t    = {}
    Qmax_fase_t    = {}

    for bus in buses_bt:
        p_carga_total = P_CARGA_NOMINAL_BARRA.get(bus, 0.0) * fator_carga_t
        q_carga_total = Q_CARGA_NOMINAL_BARRA.get(bus, 0.0) * fator_carga_t

        for ph in PHASES:
            p_carga_1f = dividir_carga_por_fase(p_carga_total, bus, ph)
            q_carga_1f = dividir_carga_por_fase(q_carga_total, bus, ph)

            P_carga_fase_t[(bus, ph)] = p_carga_1f / S_base_1ph
            Q_carga_fase_t[(bus, ph)] = q_carga_1f / S_base_1ph

            p_pv_1f = P_PV_NOMINAL.get((bus, ph), 0.0) * fator_pv_t
            P_pv_fase_t[(bus, ph)] = p_pv_1f / S_base_1ph

            if t in HORA_SEM_PV:
                Qmin_fase_t[(bus, ph)] = 0.0
                Qmax_fase_t[(bus, ph)] = 0.0
            else:
                qmin_1f, qmax_1f = calc_limites_qpv_por_fase(bus, ph, p_pv_1f)
                Qmin_fase_t[(bus, ph)] = qmin_1f / S_base_1ph
                Qmax_fase_t[(bus, ph)] = qmax_1f / S_base_1ph

    # ----------------------------------------------------------
    # Conjunto PV_BUS_PHASES ativo nesta hora
    # ----------------------------------------------------------
    PV_BUS_PHASES_t = [
        (bus, ph)
        for bus in buses_bt
        for ph in PHASES
        if pv_exists(bus, ph) and t in HORA_COM_PV
    ]

    # ----------------------------------------------------------
    # Construção do modelo Pyomo
    # ----------------------------------------------------------
    model = pyo.ConcreteModel(name=f"OPF_BT_t{t:02d}")

    # Conjuntos
    model.BUSES        = pyo.Set(initialize=[SLACK_BUS] + buses_bt)
    model.BUSES_BT     = pyo.Set(initialize=buses_bt)
    model.PHASES       = pyo.Set(initialize=PHASES)
    model.LINES_BT     = pyo.Set(initialize=LINES_BT, dimen=2)
    model.TRAFO        = pyo.Set(initialize=TRAFO_CONN, dimen=2)
    model.ALL_CONN     = model.TRAFO | model.LINES_BT
    model.PV_BUS_PHASES = pyo.Set(dimen=2, initialize=PV_BUS_PHASES_t)

    # Parâmetros
    model.R      = pyo.Param(model.ALL_CONN, initialize=R_CONN_PU, default=0.0)
    model.X      = pyo.Param(model.ALL_CONN, initialize=X_CONN_PU, default=0.0)
    model.P_LOAD = pyo.Param(model.BUSES_BT, model.PHASES, initialize=P_carga_fase_t, default=0.0)
    model.Q_LOAD = pyo.Param(model.BUSES_BT, model.PHASES, initialize=Q_carga_fase_t, default=0.0)
    model.P_PV   = pyo.Param(model.BUSES_BT, model.PHASES, initialize=P_pv_fase_t,    default=0.0)
    model.Qmax   = pyo.Param(model.BUSES_BT, model.PHASES, initialize=Qmax_fase_t,    default=0.0)
    model.Qmin   = pyo.Param(model.BUSES_BT, model.PHASES, initialize=Qmin_fase_t,    default=0.0)

    # Parâmetro com valor de Q_PV da hora anterior (para restrição de rampa)
    model.Q_PV_prev = pyo.Param(
        model.BUSES_BT, model.PHASES,
        initialize={(bus, ph): Q_PV_anterior[(bus, ph)] for bus in buses_bt for ph in PHASES},
        default=0.0
    )

    # ----------------------------------------------------------
    # Variáveis
    # ----------------------------------------------------------
    vmin      = 0.90
    vmax      = 1.10
    V_SLACK_PU = 1.0
    V_max_lin  = vmax

    model.V2 = pyo.Var(model.BUSES, model.PHASES,
                       bounds=(vmin**2, vmax**2), initialize=1.0)
    for ph in PHASES:
        model.V2[SLACK_BUS, ph].fix(V_SLACK_PU**2)

    model.P  = pyo.Var(model.ALL_CONN, model.PHASES, bounds=(-10, 10),  initialize=0.0)
    model.Q  = pyo.Var(model.ALL_CONN, model.PHASES, bounds=(-10, 10),  initialize=0.0)
    model.I2 = pyo.Var(model.ALL_CONN, model.PHASES, bounds=(0,  100),  initialize=0.001)

    model.Q_PV = pyo.Var(
        model.BUSES_BT, model.PHASES,
        bounds=lambda m, bus, ph: (m.Qmin[bus, ph], m.Qmax[bus, ph]),
        initialize=0.0
    )

    # Fixa Q_PV = 0 onde não há PV ou fora do horário solar
    for bus in buses_bt:
        for ph in PHASES:
            if t in HORA_SEM_PV or not pv_exists(bus, ph):
                model.Q_PV[bus, ph].fix(0.0)

    # Binárias e auxiliares Volt-Var (só necessárias com PV ativo)
    model.z1 = pyo.Var(model.PV_BUS_PHASES, within=pyo.Binary, initialize=0)
    model.z2 = pyo.Var(model.PV_BUS_PHASES, within=pyo.Binary, initialize=0)
    model.z3 = pyo.Var(model.PV_BUS_PHASES, within=pyo.Binary, initialize=1)
    model.z4 = pyo.Var(model.PV_BUS_PHASES, within=pyo.Binary, initialize=0)
    model.z5 = pyo.Var(model.PV_BUS_PHASES, within=pyo.Binary, initialize=0)
    model.w2 = pyo.Var(model.PV_BUS_PHASES, bounds=(0, V_max_lin), initialize=1.0)
    model.w4 = pyo.Var(model.PV_BUS_PHASES, bounds=(0, V_max_lin), initialize=1.0)

    # ----------------------------------------------------------
    # Breakpoints otimizados da curva Volt-Var por (barra, fase)
    # Bounds: [vmin, vmax] com folga minima de 0.01 pu entre consecutivos.
    # Inicializados nos valores IEEE 1547.
    # ----------------------------------------------------------
    model.V1_opt = pyo.Var(model.PV_BUS_PHASES, bounds=(vmin, vmax), initialize=V1)
    model.V2_opt = pyo.Var(model.PV_BUS_PHASES, bounds=(vmin, vmax), initialize=V2)
    model.V3_opt = pyo.Var(model.PV_BUS_PHASES, bounds=(vmin, vmax), initialize=V3)
    model.V4_opt = pyo.Var(model.PV_BUS_PHASES, bounds=(vmin, vmax), initialize=V4)

    # V^2 dos breakpoints (auxiliares para Big-M)
    model.V1_2_opt = pyo.Var(model.PV_BUS_PHASES, bounds=(vmin**2, vmax**2), initialize=V1**2)
    model.V2_2_opt = pyo.Var(model.PV_BUS_PHASES, bounds=(vmin**2, vmax**2), initialize=V2**2)
    model.V3_2_opt = pyo.Var(model.PV_BUS_PHASES, bounds=(vmin**2, vmax**2), initialize=V3**2)
    model.V4_2_opt = pyo.Var(model.PV_BUS_PHASES, bounds=(vmin**2, vmax**2), initialize=V4**2)

    # Parametros com breakpoints da hora anterior (para rampa temporal)
    model.V1_opt_prev = pyo.Param(model.PV_BUS_PHASES,
        initialize={(b,ph): BP_anterior["V1"].get((b,ph), V1) for b,ph in PV_BUS_PHASES_t}, default=V1)
    model.V2_opt_prev = pyo.Param(model.PV_BUS_PHASES,
        initialize={(b,ph): BP_anterior["V2"].get((b,ph), V2) for b,ph in PV_BUS_PHASES_t}, default=V2)
    model.V3_opt_prev = pyo.Param(model.PV_BUS_PHASES,
        initialize={(b,ph): BP_anterior["V3"].get((b,ph), V3) for b,ph in PV_BUS_PHASES_t}, default=V3)
    model.V4_opt_prev = pyo.Param(model.PV_BUS_PHASES,
        initialize={(b,ph): BP_anterior["V4"].get((b,ph), V4) for b,ph in PV_BUS_PHASES_t}, default=V4)

    # ----------------------------------------------------------
    # Restrições DistFlow
    # ----------------------------------------------------------
    def current_rule(m, i, j, ph):
        if not phase_exists_branch(i, j, ph):
            return m.I2[i, j, ph] == 0.0
        return m.I2[i, j, ph] * m.V2[i, ph] == m.P[i, j, ph]**2 + m.Q[i, j, ph]**2

    model.current_eq = pyo.Constraint(model.ALL_CONN, model.PHASES, rule=current_rule)

    def voltage_rule(m, i, j, ph):
        if not phase_exists_branch(i, j, ph):
            return pyo.Constraint.Skip
        return (m.V2[j, ph] == m.V2[i, ph]
                - 2*(m.R[i,j]*m.P[i,j,ph] + m.X[i,j]*m.Q[i,j,ph])
                + (m.R[i,j]**2 + m.X[i,j]**2)*m.I2[i,j,ph])

    model.voltage_eq = pyo.Constraint(model.ALL_CONN, model.PHASES, rule=voltage_rule)

    def balance_P_rule(m, bus, ph):
        if not phase_exists_bus(bus, ph):
            return pyo.Constraint.Skip
        i_up, j_up = UPSTREAM[bus]
        downstream = sum(m.P[i,j,ph] for (i,j) in DOWNSTREAM[bus]
                         if phase_exists_branch(i, j, ph))
        return m.P[i_up,j_up,ph] == downstream + m.P_LOAD[bus,ph] - m.P_PV[bus,ph]

    model.balance_P = pyo.Constraint(model.BUSES_BT, model.PHASES, rule=balance_P_rule)

    def balance_Q_rule(m, bus, ph):
        if not phase_exists_bus(bus, ph):
            return pyo.Constraint.Skip
        i_up, j_up = UPSTREAM[bus]
        downstream = sum(m.Q[i,j,ph] for (i,j) in DOWNSTREAM[bus]
                         if phase_exists_branch(i, j, ph))
        return m.Q[i_up,j_up,ph] == downstream + m.Q_LOAD[bus,ph] - m.Q_PV[bus,ph]

    model.balance_Q = pyo.Constraint(model.BUSES_BT, model.PHASES, rule=balance_Q_rule)

    # ----------------------------------------------------------
    # Restrição de rampa em Q_PV — acopla com hora anterior
    # Ativa a partir de t=1 e apenas nas horas com geração solar.
    # Nas horas sem PV Q_PV já está fixado em 0, então não há
    # necessidade de rampa (a transição noite→dia é livre).
    # ----------------------------------------------------------
    if t > 0 and t in HORA_COM_PV:
        def rampa_lo_rule(m, bus, ph):
            if not pv_exists(bus, ph):
                return pyo.Constraint.Skip
            q_prev = m.Q_PV_prev[bus, ph]
            delta  = DELTA_Q_MAX_FRAC * m.Qmax[bus, ph]
            return m.Q_PV[bus, ph] >= q_prev - delta

        def rampa_up_rule(m, bus, ph):
            if not pv_exists(bus, ph):
                return pyo.Constraint.Skip
            q_prev = m.Q_PV_prev[bus, ph]
            delta  = DELTA_Q_MAX_FRAC * m.Qmax[bus, ph]
            return m.Q_PV[bus, ph] <= q_prev + delta

        model.rampa_lo = pyo.Constraint(model.BUSES_BT, model.PHASES, rule=rampa_lo_rule)
        model.rampa_up = pyo.Constraint(model.BUSES_BT, model.PHASES, rule=rampa_up_rule)

        print(f"  Rampa ΔQ_max = ±{DELTA_Q_MAX_FRAC*100:.0f}% de Qmax  (acoplado a t={t-1:02d}h)")

    # ----------------------------------------------------------
    # Restrições Volt-Var — só nas horas com PV
    # Breakpoints V1_opt..V4_opt são agora VARIÁVEIS de decisão,
    # permitindo que o solver ajuste a forma da curva para minimizar
    # perdas. As restrições Big-M usam V1_2_opt..V4_2_opt (= Vi_opt^2).
    # ----------------------------------------------------------
    if t in HORA_COM_PV:

        # Ordenação V1 < V2 < V3 < V4 (folga mínima 0.01 pu)
        model.ord_v12 = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.V1_opt[b,ph] <= m.V2_opt[b,ph] - 0.01)
        model.ord_v23 = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.V2_opt[b,ph] <= m.V3_opt[b,ph] - 0.01)
        model.ord_v34 = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.V3_opt[b,ph] <= m.V4_opt[b,ph] - 0.01)

        # V^2 dos breakpoints = Vi_opt * Vi_opt
        model.v1_sq = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.V1_2_opt[b,ph] == m.V1_opt[b,ph]**2)
        model.v2_sq = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.V2_2_opt[b,ph] == m.V2_opt[b,ph]**2)
        model.v3_sq = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.V3_2_opt[b,ph] == m.V3_opt[b,ph]**2)
        model.v4_sq = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.V4_2_opt[b,ph] == m.V4_opt[b,ph]**2)

        # SOS1 — exatamente uma zona ativa por (barra, fase)
        def sos1_rule(m, bus, ph):
            return m.z1[bus,ph]+m.z2[bus,ph]+m.z3[bus,ph]+m.z4[bus,ph]+m.z5[bus,ph] == 1
        model.sos1 = pyo.Constraint(model.PV_BUS_PHASES, rule=sos1_rule)

        # Big-M com breakpoints otimizados
        model.z1_up = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.V2[b,ph] <= m.V1_2_opt[b,ph] + M_bigm*(1-m.z1[b,ph]))
        model.z2_lo = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.V2[b,ph] >= m.V1_2_opt[b,ph] - M_bigm*(1-m.z2[b,ph]))
        model.z2_up = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.V2[b,ph] <= m.V2_2_opt[b,ph] + M_bigm*(1-m.z2[b,ph]))
        model.z3_lo = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.V2[b,ph] >= m.V2_2_opt[b,ph] - M_bigm*(1-m.z3[b,ph]))
        model.z3_up = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.V2[b,ph] <= m.V3_2_opt[b,ph] + M_bigm*(1-m.z3[b,ph]))
        model.z4_lo = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.V2[b,ph] >= m.V3_2_opt[b,ph] - M_bigm*(1-m.z4[b,ph]))
        model.z4_up = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.V2[b,ph] <= m.V4_2_opt[b,ph] + M_bigm*(1-m.z4[b,ph]))
        model.z5_lo = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.V2[b,ph] >= m.V4_2_opt[b,ph] - M_bigm*(1-m.z5[b,ph]))

        # Linearizacao w2, w4 (igual a antes)
        model.w2_up_bm   = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.w2[b,ph] <= V_max_lin*m.z2[b,ph])
        model.w2_sq_lo   = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.w2[b,ph]**2 >= m.V2[b,ph] - M_bigm*(1-m.z2[b,ph]))
        model.w2_sq_up   = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.w2[b,ph]**2 <= m.V2[b,ph] + M_bigm*(1-m.z2[b,ph]))
        model.w2_sq_zero = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.w2[b,ph]**2 <= (V_max_lin**2)*m.z2[b,ph])
        model.w4_up_bm   = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.w4[b,ph] <= V_max_lin*m.z4[b,ph])
        model.w4_sq_lo   = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.w4[b,ph]**2 >= m.V2[b,ph] - M_bigm*(1-m.z4[b,ph]))
        model.w4_sq_up   = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.w4[b,ph]**2 <= m.V2[b,ph] + M_bigm*(1-m.z4[b,ph]))
        model.w4_sq_zero = pyo.Constraint(model.PV_BUS_PHASES,
            rule=lambda m,b,ph: m.w4[b,ph]**2 <= (V_max_lin**2)*m.z4[b,ph])

        # Curva Volt-Var com breakpoints otimizados
        # slope2 = -Qmax / (V2_opt - V1_opt)  — slope é agora nao-linear (divisao de variaveis)
        # Reformulado: Q_PV*(V2_opt-V1_opt) = expressao linear nos termos z, w, V_opt
        def volt_var_rule(m, bus, ph):
            qmax = m.Qmax[bus, ph]
            dV2  = m.V2_opt[bus,ph] - m.V1_opt[bus,ph]   # > 0 pela ord_v12
            dV4  = m.V4_opt[bus,ph] - m.V3_opt[bus,ph]   # > 0 pela ord_v34
            # Forma multiplicada para evitar divisao de variaveis:
            # Q_PV = z1*Qmax + z2*(Qmax - Qmax/dV2*(w2-V1_opt*z2))
            #                + (-Qmax/dV4)*(w4 - V3_opt*z4) - z5*Qmax
            # Multiplicando por dV2 (zona 2) e dV4 (zona 4) isoladamente:
            # Equacao equivalente sem divisao:
            # Q_PV*dV2*dV4 = z1*Qmax*dV2*dV4
            #              + [z2*Qmax*dV2 - Qmax*(w2-V1_opt*z2)]*dV4
            #              + [-Qmax*(w4-V3_opt*z4)]*dV2
            #              - z5*Qmax*dV2*dV4
            lhs = m.Q_PV[bus,ph] * dV2 * dV4
            t1  = m.z1[bus,ph]  * qmax * dV2 * dV4
            t2  = (m.z2[bus,ph] * qmax * dV2
                   - qmax * (m.w2[bus,ph] - m.V1_opt[bus,ph]*m.z2[bus,ph])) * dV4
            t4  = -qmax * (m.w4[bus,ph] - m.V3_opt[bus,ph]*m.z4[bus,ph]) * dV2
            t5  = -m.z5[bus,ph] * qmax * dV2 * dV4
            return lhs == t1 + t2 + t4 + t5

        model.volt_var = pyo.Constraint(model.PV_BUS_PHASES, rule=volt_var_rule)

        # ----------------------------------------------------------
        # Rampa temporal dos breakpoints — limita variacao entre horas
        # Mesma fracao DELTA_Q_MAX_FRAC aplicada ao intervalo de tensao
        # ----------------------------------------------------------
        if t > 0:
            DELTA_BP = 0.05   # variacao maxima de cada breakpoint por hora (pu)
            model.bp_v1_lo = pyo.Constraint(model.PV_BUS_PHASES,
                rule=lambda m,b,ph: m.V1_opt[b,ph] >= m.V1_opt_prev[b,ph] - DELTA_BP)
            model.bp_v1_up = pyo.Constraint(model.PV_BUS_PHASES,
                rule=lambda m,b,ph: m.V1_opt[b,ph] <= m.V1_opt_prev[b,ph] + DELTA_BP)
            model.bp_v2_lo = pyo.Constraint(model.PV_BUS_PHASES,
                rule=lambda m,b,ph: m.V2_opt[b,ph] >= m.V2_opt_prev[b,ph] - DELTA_BP)
            model.bp_v2_up = pyo.Constraint(model.PV_BUS_PHASES,
                rule=lambda m,b,ph: m.V2_opt[b,ph] <= m.V2_opt_prev[b,ph] + DELTA_BP)
            model.bp_v3_lo = pyo.Constraint(model.PV_BUS_PHASES,
                rule=lambda m,b,ph: m.V3_opt[b,ph] >= m.V3_opt_prev[b,ph] - DELTA_BP)
            model.bp_v3_up = pyo.Constraint(model.PV_BUS_PHASES,
                rule=lambda m,b,ph: m.V3_opt[b,ph] <= m.V3_opt_prev[b,ph] + DELTA_BP)
            model.bp_v4_lo = pyo.Constraint(model.PV_BUS_PHASES,
                rule=lambda m,b,ph: m.V4_opt[b,ph] >= m.V4_opt_prev[b,ph] - DELTA_BP)
            model.bp_v4_up = pyo.Constraint(model.PV_BUS_PHASES,
                rule=lambda m,b,ph: m.V4_opt[b,ph] <= m.V4_opt_prev[b,ph] + DELTA_BP)
            print(f"  Rampa breakpoints ΔV = ±{DELTA_BP:.2f} pu  (acoplado a t={t-1:02d}h)")

    # ----------------------------------------------------------
    # Função objetivo — perdas ativas da hora t
    # ----------------------------------------------------------
    def obj_rule(m):
        return sum(
            m.R[i,j] * m.I2[i,j,ph]
            for (i,j) in m.ALL_CONN
            for ph in m.PHASES
            if phase_exists_branch(i, j, ph)
        )

    model.obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

    # ----------------------------------------------------------
    # Seleção e execução do solver
    # ----------------------------------------------------------
    if t in HORA_SEM_PV:
        print(f"  Modo: NLP puro (sem binárias livres)")
        solver, solver_nome = configurar_solver_nlp()
    else:
        print(f"  Modo: MINLP (binárias Volt-Var ativas)")
        solver, solver_nome = configurar_solver_minlp()

    results_solver = solver.solve(model, tee=False)
    status = results_solver.solver.termination_condition
    print(f"  Solver: {solver_nome.upper()} | Status: {status}")

    # ----------------------------------------------------------
    # Extração de resultados
    # ----------------------------------------------------------
    resolvido = status in (
        pyo.TerminationCondition.optimal,
        pyo.TerminationCondition.locallyOptimal,
        pyo.TerminationCondition.feasible,
    )

    if resolvido:
        losses_pu = sum(
            safe_val(model.R[i,j], 0.0) * safe_val(model.I2[i,j,ph], 0.0)
            for (i,j) in model.ALL_CONN
            for ph in model.PHASES
            if phase_exists_branch(i, j, ph)
        )
        losses_kW = losses_pu * S_base_1ph

        V_bus_phase_pu = {
            (bus, ph): np.sqrt(max(0.0, safe_val(model.V2[bus, ph], 0.0)))
            for bus in model.BUSES for ph in PHASES
        }
        V_max_bt = max(V_bus_phase_pu[(b,ph)] for b in buses_bt for ph in PHASES
                       if phase_exists_bus(b, ph))
        V_min_bt = min(V_bus_phase_pu[(b,ph)] for b in buses_bt for ph in PHASES
                       if phase_exists_bus(b, ph))

        P_trafo_kW = {
            ph: sum(safe_val(model.P[SLACK_BUS, root, ph], 0.0) * S_base_1ph
                    for root in ROOTS_BT if phase_exists_branch(SLACK_BUS, root, ph))
            for ph in PHASES
        }
        Q_trafo_kVAr = {
            ph: sum(safe_val(model.Q[SLACK_BUS, root, ph], 0.0) * S_base_1ph
                    for root in ROOTS_BT if phase_exists_branch(SLACK_BUS, root, ph))
            for ph in PHASES
        }

        Q_pv_kVAr = {
            (bus, ph): safe_val(model.Q_PV[bus, ph], 0.0) * S_base_1ph
            for bus in buses_bt for ph in PHASES
        }
        Q_pv_total = sum(Q_pv_kVAr.values())

        print(f"  Perdas={losses_kW:.4f} kW | V_max={V_max_bt:.4f} | "
              f"V_min={V_min_bt:.4f} | Q_PV_total={Q_pv_total:.3f} kVAr")

        # Monta dicionário de resultados desta hora
        result_dict = {
            "hora":              t,
            "fator_carga":       fator_carga_t,
            "fator_pv":          fator_pv_t,
            "losses_kW":         round(losses_kW, 5),
            "V_max_pu":          round(V_max_bt, 5),
            "V_min_pu":          round(V_min_bt, 5),
            "P_slack_total_kW":  round(sum(P_trafo_kW.values()), 5),
            "Q_slack_total_kVAr":round(sum(Q_trafo_kVAr.values()), 5),
            "Q_pv_total_kVAr":   round(Q_pv_total, 5),
            "status":            str(status),
        }
        for bus in buses_bt:
            for ph in PHASES:
                result_dict[f"V_{bus}_{ph}_pu"]     = round(V_bus_phase_pu[(bus, ph)], 6)
                result_dict[f"Qpv_{bus}_{ph}_kVAr"] = round(Q_pv_kVAr[(bus, ph)], 5)

        # Salva breakpoints otimizados no resultado (so horas com PV)
        if t in HORA_COM_PV:
            for b, ph in PV_BUS_PHASES_t:
                result_dict[f"V1_{b}_{ph}"] = round(safe_val(model.V1_opt[b,ph], V1), 4)
                result_dict[f"V2_{b}_{ph}"] = round(safe_val(model.V2_opt[b,ph], V2), 4)
                result_dict[f"V3_{b}_{ph}"] = round(safe_val(model.V3_opt[b,ph], V3), 4)
                result_dict[f"V4_{b}_{ph}"] = round(safe_val(model.V4_opt[b,ph], V4), 4)

        resultados_lista.append(result_dict)

        # -------------------------------------------------------
        # Atualiza memória inter-hora com Q_PV desta hora
        # Nas horas sem PV Q_PV = 0 → estado da memória reseta,
        # o que é fisicamente correto (inversores desligados à noite).
        # -------------------------------------------------------
        for bus in buses_bt:
            for ph in PHASES:
                Q_PV_anterior[(bus, ph)] = safe_val(model.Q_PV[bus, ph], 0.0)

        # Atualiza breakpoints da hora anterior (so para horas com PV)
        if t in HORA_COM_PV:
            for b, ph in PV_BUS_PHASES_t:
                BP_anterior["V1"][(b,ph)] = safe_val(model.V1_opt[b,ph], V1)
                BP_anterior["V2"][(b,ph)] = safe_val(model.V2_opt[b,ph], V2)
                BP_anterior["V3"][(b,ph)] = safe_val(model.V3_opt[b,ph], V3)
                BP_anterior["V4"][(b,ph)] = safe_val(model.V4_opt[b,ph], V4)

    else:
        print(f"  ✗ Sem solução ótima — mantendo Q_PV_anterior do instante t={t-1:02d}h")
        result_dict = {
            "hora": t, "fator_carga": fator_carga_t, "fator_pv": fator_pv_t,
            "losses_kW": np.nan, "V_max_pu": np.nan, "V_min_pu": np.nan,
            "P_slack_total_kW": np.nan, "Q_slack_total_kVAr": np.nan,
            "Q_pv_total_kVAr": np.nan, "status": str(status),
        }
        resultados_lista.append(result_dict)
        # Q_PV_anterior NÃO é atualizado — mantém o valor da última hora resolvida



HORA t=00h  |  fator_carga=0.40  |  fator_pv=0.0000
  Modo: NLP puro (sem binárias livres)
  Solver: IPOPT | Status: optimal
  Perdas=8.4454 kW | V_max=1.0000 | V_min=0.9890 | Q_PV_total=0.000 kVAr

HORA t=01h  |  fator_carga=0.35  |  fator_pv=0.0000
  Modo: NLP puro (sem binárias livres)
  Solver: IPOPT | Status: optimal
  Perdas=6.4636 kW | V_max=1.0000 | V_min=0.9903 | Q_PV_total=0.000 kVAr

HORA t=02h  |  fator_carga=0.30  |  fator_pv=0.0000
  Modo: NLP puro (sem binárias livres)
  Solver: IPOPT | Status: optimal
  Perdas=4.7470 kW | V_max=1.0000 | V_min=0.9917 | Q_PV_total=0.000 kVAr

HORA t=03h  |  fator_carga=0.28  |  fator_pv=0.0000
  Modo: NLP puro (sem binárias livres)
  Solver: IPOPT | Status: optimal
  Perdas=4.1346 kW | V_max=1.0000 | V_min=0.9923 | Q_PV_total=0.000 kVAr

HORA t=04h  |  fator_carga=0.28  |  fator_pv=0.0000
  Modo: NLP puro (sem binárias livres)
  Solver: IPOPT | Status: optimal
  Perdas=4.1346 kW | V_max=1.0000 | V_min=0.9923 | Q_PV_total=0.000 kVAr

HORA

### Resultados

In [44]:
# ============================================================
# CONSOLIDAÇÃO DOS RESULTADOS
# ============================================================
results_df = pd.DataFrame(resultados_lista).sort_values("hora").reset_index(drop=True)

print(f"\n{'='*70}")
print("RESULTADOS CONSOLIDADOS — 24 HORAS")
print(f"{'='*70}")
cols_main = ["hora","fator_carga","fator_pv","losses_kW",
             "V_max_pu","V_min_pu","Q_pv_total_kVAr","status"]
print(results_df[cols_main].to_string(index=False))

print(f"\n  Perdas totais (kWh/dia): {results_df['losses_kW'].sum():.4f}")
print(f"  V_max global:            {results_df['V_max_pu'].max():.5f} pu")
print(f"  V_min global:            {results_df['V_min_pu'].min():.5f} pu")
print(f"  Q_PV total (kVArh/dia):  {results_df['Q_pv_total_kVAr'].sum():.3f}")


RESULTADOS CONSOLIDADOS — 24 HORAS
 hora  fator_carga     fator_pv  losses_kW  V_max_pu  V_min_pu  Q_pv_total_kVAr  status
    0         0.40 0.000000e+00    8.44540       1.0   0.98896              0.0 optimal
    1         0.35 0.000000e+00    6.46361       1.0   0.99034              0.0 optimal
    2         0.30 0.000000e+00    4.74703       1.0   0.99172              0.0 optimal
    3         0.28 0.000000e+00    4.13458       1.0   0.99227              0.0 optimal
    4         0.28 0.000000e+00    4.13458       1.0   0.99227              0.0 optimal
    5         0.32 0.000000e+00    5.40186       1.0   0.99117              0.0 optimal
    6         0.45 0.000000e+00   10.69268       1.0   0.98757              0.0 optimal
    7         0.65 0.000000e+00   22.34281       1.0   0.98204              0.0 optimal
    8         0.75 2.817326e-01   18.04056       1.0   0.98319             -0.0 optimal
    9         0.70 5.406408e-01    8.06851       1.0   0.98816             -0.0 opti

In [45]:
# # --------------------------------------------------------
# # Dicionário de resultados
# # --------------------------------------------------------
# result_dict = {
#     "load_factor": FATOR_CARGA,
#     "pv_factor": FATOR_PV,
#     "V_max_pu": round(V_max_bt, 6),
#     "V_min_pu": round(V_min_bt, 6),
#     "losses_total_kW": round(losses_kW_total, 6),
#     "P_slack_total_kW": round(P_trafo_total_kW, 6),
#     "Q_slack_total_kVAr": round(Q_trafo_total_kVAr, 6),
#     "Q_pv_total_kVAr": round(Q_pv_total_kVAr, 6),
# }

# # Tensões por barra-fase
# for bus in model.BUSES:
#     for ph in PHASES:
#         if bus != SLACK_BUS and not phase_exists_bus(bus, ph):
#             continue

#         vpu = V_bus_phase_pu[(bus, ph)]
#         result_dict[f"V_{bus}_{ph}_pu"] = round(vpu, 6)
#         result_dict[f"V_{bus}_{ph}_Vfn"] = round(vpu * V_BT_LN * 1000, 3)
#         result_dict[f"V_{bus}_{ph}_Vff"] = round(vpu * V_BT_LL * 1000, 3)

# # Q_PV por barra-fase
# for bus in buses_bt:
#     q_total_bus = 0.0
#     for ph in PHASES:
#         qpv = Q_pv_bus_phase_kVAr[(bus, ph)]
#         q_total_bus += qpv
#         result_dict[f"Qpv_{bus}_{ph}_kVAr"] = round(qpv, 6)

#     result_dict[f"Qpv_{bus}_total_kVAr"] = round(q_total_bus, 6)

# # potência do slack por fase
# for ph in PHASES:
#     result_dict[f"P_slack_{ph}_kW"] = round(P_trafo_phase_kW[ph], 6)
#     result_dict[f"Q_slack_{ph}_kVAr"] = round(Q_trafo_phase_kVAr[ph], 6)

# resultados_lista.append(result_dict)

# print("\n✓ Cenário resolvido com sucesso.")

# if status == "optimal":
# resultados_lista.append(result_dict)
# print("\n✓ Cenário resolvido com sucesso.")
# else:
# print(f"\n✗ Sem solução ótima: {status}")
# print("  → Revisar dados, escalas em pu, limites de tensão e bounds de P/Q/I2.")

#### Consolidados (DF)

In [46]:
# # --------------------------------------------------------
# # Dicionário de resultados
# # --------------------------------------------------------
# result_dict = {
#     "load_factor": FATOR_CARGA,
#     "pv_factor": FATOR_PV,
#     "V_max_pu": round(V_max_bt, 6),
#     "V_min_pu": round(V_min_bt, 6),
#     "losses_total_kW": round(losses_kW_total, 6),
#     "P_slack_total_kW": round(P_trafo_total_kW, 6),
#     "Q_slack_total_kVAr": round(Q_trafo_total_kVAr, 6),
#     "Q_pv_total_kVAr": round(Q_pv_total_kVAr, 6),
# }

# # Tensões por barra-fase
# for bus in model.BUSES:
#     for ph in PHASES:
#         if bus != SLACK_BUS and not phase_exists_bus(bus, ph):
#             continue

#         vpu = V_bus_phase_pu[(bus, ph)]
#         result_dict[f"V_{bus}_{ph}_pu"] = round(vpu, 6)
#         result_dict[f"V_{bus}_{ph}_Vfn"] = round(vpu * V_BT_LN * 1000, 3)
#         result_dict[f"V_{bus}_{ph}_Vff"] = round(vpu * V_BT_LL * 1000, 3)

# # Q_PV por barra-fase
# for bus in buses_bt:
#     q_total_bus = 0.0
#     for ph in PHASES:
#         qpv = Q_pv_bus_phase_kVAr[(bus, ph)]
#         q_total_bus += qpv
#         result_dict[f"Qpv_{bus}_{ph}_kVAr"] = round(qpv, 6)

#     result_dict[f"Qpv_{bus}_total_kVAr"] = round(q_total_bus, 6)

# # potência do slack por fase
# for ph in PHASES:
#     result_dict[f"P_slack_{ph}_kW"] = round(P_trafo_phase_kW[ph], 6)
#     result_dict[f"Q_slack_{ph}_kVAr"] = round(Q_trafo_phase_kVAr[ph], 6)

# resultados_lista.append(result_dict)

# print("\n✓ Cenário resolvido com sucesso.")

# else:
# print(f"\n✗ Sem solução ótima: {status}")
# print("  → Revisar dados, escalas em pu, limites de tensão e bounds de P/Q/I2.")

In [47]:
# # ============================================================
# # RESULTADOS CONSOLIDADOS
# # ============================================================
# if resultados_lista:
#     results_df = pd.DataFrame(resultados_lista)

#     print(f"\n{'='*90}")
#     print("RESULTADOS CONSOLIDADOS")
#     print(f"{'='*90}")

#     cols_main = [
#         "load_factor", "pv_factor",
#         "V_max_pu", "V_min_pu", "losses_total_kW",
#         "P_slack_total_kW", "Q_slack_total_kVAr", "Q_pv_total_kVAr"
#     ]
#     print(results_df[cols_main].to_string(index=False))

#     print(f"\n{'='*90}")
#     print("POTÊNCIA DO SLACK POR FASE")
#     print(f"{'='*90}")
#     cols_slack = ["load_factor", "pv_factor"] + \
#                  [f"P_slack_{ph}_kW" for ph in PHASES] + \
#                  [f"Q_slack_{ph}_kVAr" for ph in PHASES]
#     print(results_df[cols_slack].to_string(index=False))

# else:
#     print("\n⚠ Sem resultados válidos.")

In [48]:
# print("\nTensões por barra-fase:")
# for bus in model.BUSES:
#     vals = []
#     for ph in PHASES:
#         if bus != SLACK_BUS and not phase_exists_bus(bus, ph):
#             continue
#         v = np.sqrt(max(0.0, safe_val(model.V2[bus, ph], 0.0)))
#         vals.append(f"{ph}={v:.4f}")
#     print(f"  Barra {bus}: " + " | ".join(vals))

# print("\nQ_PV por barra-fase [pu]:")
# for bus in model.BUSES_BT:
#     vals = []
#     for ph in PHASES:
#         qpv = safe_val(model.Q_PV[bus, ph], 0.0)
#         if abs(qpv) < 1e-10 and abs(safe_val(model.Qmax[bus, ph], 0.0)) < 1e-10:
#             continue
#         vals.append(f"{ph}={qpv:+.4f}")
#     if vals:
#         print(f"  Barra {bus}: " + " | ".join(vals))

# print(f"\nPerdas totais = {losses_pu_total:.6f} pu ({losses_kW_total:.4f} kW na base 1φ)")

#### Gráficos

In [49]:
# import math
# import numpy as np
# import matplotlib.pyplot as plt

# # ============================================================
# # CURVA VOLT-VAR
# # ============================================================
# V1, V2, V3, V4 = 0.92, 0.98, 1.02, 1.08

# def curva_voltvar(Qmax, n_pts=500):
#     V_vec = np.linspace(0.88, 1.14, n_pts)
#     Q_vec = np.zeros(n_pts)

#     if abs(Qmax) < 1e-12:
#         return V_vec, Q_vec

#     slope2 = -Qmax / (V2 - V1)
#     slope4 = -Qmax / (V4 - V3)

#     for k, v in enumerate(V_vec):
#         if v < V1:
#             Q_vec[k] = Qmax
#         elif v < V2:
#             Q_vec[k] = Qmax + slope2 * (v - V1)
#         elif v <= V3:
#             Q_vec[k] = 0.0
#         elif v <= V4:
#             Q_vec[k] = slope4 * (v - V3)
#         else:
#             Q_vec[k] = -Qmax

#     return V_vec, Q_vec


# def safe_val(x, default=np.nan):
#     try:
#         v = pyo.value(x)
#         if v is None:
#             return default
#         return float(v)
#     except Exception:
#         return default


# # ============================================================
# # FUNÇÃO PRINCIPAL
# # ============================================================
# def plot_voltvar_top_n_barras(
#     model,
#     buses_bt,
#     PHASES,
#     S_base_1ph,
#     BUS_TYPE,
#     N=6,
#     scenario_name="cenário",
#     load_factor=None,
#     pv_factor=None,
#     save_path="voltvar_topN_barras.png"
# ):
#     """
#     Plota as N barras com maior |Q_PV| total.
#     Cada subplot mostra as 3 fases da barra.
#     """

#     # --------------------------------------------------------
#     # Monta dados por barra-fase
#     # --------------------------------------------------------
#     dados_barras = {}
#     q_total_abs_por_barra = {}

#     for bus in buses_bt:
#         dados_barras[bus] = {}
#         q_total_abs = 0.0

#         for ph in PHASES:
#             q_kvar = safe_val(model.Q_PV[bus, ph], 0.0) * S_base_1ph
#             qmax_kvar = safe_val(model.Qmax[bus, ph], 0.0) * S_base_1ph
#             v_pu = np.sqrt(max(0.0, safe_val(model.V2[bus, ph], 0.0)))

#             dados_barras[bus][ph] = {
#                 "V": v_pu,
#                 "Q": q_kvar,
#                 "Qmax": qmax_kvar,
#                 "tipo": BUS_TYPE.get(bus, "unknown"),
#             }

#             q_total_abs += abs(q_kvar)

#         q_total_abs_por_barra[bus] = q_total_abs

#     # --------------------------------------------------------
#     # Seleciona as N barras com maior |Q_PV| total
#     # --------------------------------------------------------
#     barras_ordenadas = sorted(
#         q_total_abs_por_barra.keys(),
#         key=lambda b: q_total_abs_por_barra[b],
#         reverse=True
#     )

#     barras_top = [b for b in barras_ordenadas if q_total_abs_por_barra[b] > 1e-10][:N]

#     if not barras_top:
#         print("Nenhuma barra com |Q_PV| relevante para plotar.")
#         return

#     # --------------------------------------------------------
#     # Layout dinâmico
#     # --------------------------------------------------------
#     n_plots = len(barras_top)
#     ncols = 2 if n_plots > 1 else 1
#     nrows = math.ceil(n_plots / ncols)

#     fig, axes = plt.subplots(nrows, ncols, figsize=(7*ncols, 4.8*nrows))
#     if n_plots == 1:
#         axes = np.array([axes])
#     axes_flat = np.array(axes).flatten()

#     titulo = f"Curvas Volt-Var — Top {n_plots} barras com maior |Q_PV|"
#     if load_factor is not None and pv_factor is not None:
#         titulo += f"\n{scenario_name} · fator_carga={load_factor} · fator_PV={pv_factor}"
#     else:
#         titulo += f"\n{scenario_name}"

#     fig.suptitle(titulo, fontsize=13, fontweight="bold", y=0.98)

#     phase_colors = {"a": "tab:blue", "b": "tab:orange", "c": "tab:green"}

#     # --------------------------------------------------------
#     # Plota
#     # --------------------------------------------------------
#     for idx, bus in enumerate(barras_top):
#         ax = axes_flat[idx]

#         # fundo das zonas
#         ax.axvspan(0.88, V1, color="#d4edda", alpha=0.35)
#         ax.axvspan(V1, V2, color="#d4edda", alpha=0.20)
#         ax.axvspan(V2, V3, color="#e9ecef", alpha=0.40)
#         ax.axvspan(V3, V4, color="#fde8d8", alpha=0.20)
#         ax.axvspan(V4, 1.14, color="#fde8d8", alpha=0.35)

#         for vx, lbl in [(V1, "V1"), (V2, "V2"), (V3, "V3"), (V4, "V4")]:
#             ax.axvline(vx, color="gray", linewidth=0.8, linestyle="--")
#             ax.text(
#                 vx, 0.95, lbl,
#                 transform=ax.get_xaxis_transform(),
#                 ha="center", va="top",
#                 fontsize=8, color="gray"
#             )

#         ax.axhline(0, color="gray", linewidth=0.8)

#         qmax_global = max(dados_barras[bus][ph]["Qmax"] for ph in PHASES)
#         qmax_global = max(qmax_global, 0.1)

#         for ph in PHASES:
#             d = dados_barras[bus][ph]
#             V_op = d["V"]
#             Q_op = d["Q"]
#             Qmax = d["Qmax"]

#             # só plota fase com PV/capacidade
#             if abs(Qmax) < 1e-12 and abs(Q_op) < 1e-12:
#                 continue

#             V_vec, Q_vec = curva_voltvar(Qmax)

#             ax.plot(V_vec, Q_vec, linewidth=2, color=phase_colors[ph], label=f"fase {ph}")
#             ax.plot(V_op, Q_op, "o", color=phase_colors[ph], markersize=8)
#             ax.plot([V_op, V_op], [-1.2*qmax_global, Q_op], linestyle=":", color=phase_colors[ph], alpha=0.7)
#             ax.plot([0.88, V_op], [Q_op, Q_op], linestyle=":", color=phase_colors[ph], alpha=0.7)

#             ax.annotate(
#                 f"{ph}: V={V_op:.4f} pu\nQ={Q_op:+.2f} kVAr",
#                 xy=(V_op, Q_op),
#                 xytext=(V_op + 0.003, Q_op + 0.08*qmax_global),
#                 fontsize=7.5,
#                 color=phase_colors[ph],
#                 bbox=dict(
#                     boxstyle="round,pad=0.25",
#                     facecolor="white",
#                     edgecolor=phase_colors[ph],
#                     alpha=0.85
#                 )
#             )

#         tipo_label = "3Φ" if BUS_TYPE.get(bus) == "three_phase" else "1Φ"
#         q_total_barra = sum(abs(dados_barras[bus][ph]["Q"]) for ph in PHASES)

#         ax.set_title(
#             f"Barra {bus} [{tipo_label}] | Σ|Q_PV|={q_total_barra:.2f} kVAr",
#             fontsize=10, fontweight="bold"
#         )
#         ax.set_xlim(0.88, 1.14)
#         ax.set_ylim(-1.25*qmax_global, 1.25*qmax_global)
#         ax.set_xlabel("Tensão (pu)")
#         ax.set_ylabel("Q_PV (kVAr)")
#         ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.5)
#         ax.legend(fontsize=8)

#     # esconde subplots sobrando
#     for j in range(n_plots, len(axes_flat)):
#         axes_flat[j].set_visible(False)

#     plt.tight_layout(rect=[0, 0, 1, 0.96])
#     plt.savefig(save_path, dpi=150, bbox_inches="tight")
#     plt.show()

#     # print(f"✓ Gráfico salvo em: {save_path}")
#     # print("Barras plotadas:", barras_top)

In [50]:
# plot_voltvar_top_n_barras(
#     model=model,
#     buses_bt=buses_bt,
#     PHASES=PHASES,
#     S_base_1ph=S_base_1ph,
#     BUS_TYPE=BUS_TYPE,
#     N=8,
#     scenario_name="Caso base",
#     load_factor=FATOR_CARGA,
#     pv_factor=FATOR_PV,
#     save_path="voltvar_top8_barras.png"
# )